# Analisi di immagini astronomiche in formato FITS

Il formato **FITS** (Flexible Image Transport System) è lo standard internazionale per la memorizzazione e lo scambio di dati astronomici. Creato negli anni '80, è utilizzato da tutti i principali osservatori e missioni spaziali (Hubble, Chandra, JWST, ecc.).

Caratteristiche principali:
- **Header**: metadati del file in formato ASCII (parole chiave = valore)
- **Data**: array multidimensionale contenente l'immagine o lo spettro
- Supporta più estensioni (HDU - Header Data Unit)

In questo notebook analizzeremo un'immagine FITS della galassia **M87**, nota per ospitare il primo buco nero mai fotografato (M87*).

## 1. Verifica del file e import librerie

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

%matplotlib inline

def verifica_file(filepath: str) -> bool:
    """Verifica se un file esiste e ne stampa il percorso assoluto.

    Args:
        filepath: Percorso relativo del file

    Returns:
        True se il file esiste, False altrimenti
    """
    print(f"Directory corrente: {os.getcwd()}")
    path_completo = os.path.abspath(filepath)
    esiste = os.path.exists(filepath)
    print(f"File: {path_completo}")
    print(f"Esiste: {esiste}")
    return esiste

file_path = './M87_NEW.fits'
if not verifica_file(file_path):
    print("ATTENZIONE: Il file M87_NEW.fits non è stato trovato.")

## 2. Apertura e ispezione del file FITS

Usiamo `fits.open()` per caricare il file. Il parametro `memmap=False` evita problemi di memory mapping su alcuni sistemi. La struttura tipica di un file FITS include:
- **PrimaryHDU**: la prima unità, contenente l'array principale
- **ImageHDU** / **BinTableHDU**: estensioni aggiuntive

Il metodo `.info()` mostra la struttura completa del file.

In [ ]:
def apri_fits(filepath: str, memmap: bool = False):
    """Apre un file FITS e ne mostra la struttura.

    Args:
        filepath: Percorso del file FITS
        memmap: Se usare memory mapping (default: False per compatibilità)

    Returns:
        Tuple (hdul, header, data) oppure (None, None, None) in caso di errore
    """
    if not os.path.exists(filepath):
        print(f"ERRORE: File '{filepath}' non trovato.")
        return None, None, None

    try:
        hdul = fits.open(filepath, memmap=memmap)
        print("Struttura del file FITS:")
        hdul.info()
        header = hdul[0].header
        dati = hdul[0].data
        return hdul, header, dati
    except Exception as e:
        print(f"ERRORE nell'apertura del file: {e}")
        return None, None, None

hdul, header, image_data = apri_fits(file_path)
if hdul is not None:
    hdul.close()

## 3. Esplorazione dei dati dell'immagine

I dati letti da un file FITS sono un array NumPy. Possiamo esaminarne il tipo, la forma e le statistiche di base.

In [ ]:
def esplora_dati(dati: np.ndarray):
    """Analizza le proprietà di base di un array immagine FITS.

    Args:
        dati: Array numpy contenente l'immagine

    Returns:
        Dizionario con le statistiche calcolate
    """
    if dati is None:
        print("Nessun dato disponibile.")
        return None

    print(f"Tipo dato: {dati.dtype}")
    print(f"Dimensioni: {dati.shape}")
    print(f"\nValori campione (primi 3x3):\n{dati[:3, :3]}")

    stats = {
        'min': float(np.min(dati)),
        'max': float(np.max(dati)),
        'mean': float(np.mean(dati)),
        'std': float(np.std(dati)),
        'mediana': float(np.median(dati))
    }

    print(f"\nStatistiche immagine:")
    for chiave, valore in stats.items():
        print(f"  {chiave.capitalize()}: {valore:.2f}")

    return stats

stats = esplora_dati(image_data)

## 4. Visualizzazione dell'immagine

Visualizziamo l'immagine FITS con una mappa di colori a falsi colori per evidenziare le strutture di M87. La colorbar mostra la corrispondenza tra colore e intensità dei pixel.

In [ ]:
def visualizza_immagine(dati: np.ndarray, cmap: str = 'inferno', titolo: str = 'M87 - Immagine FITS'):
    """Visualizza un'immagine FITS con barra dei colori.

    Args:
        dati: Array numpy dell'immagine
        cmap: Mappa di colori matplotlib
        titolo: Titolo del grafico
    """
    if dati is None:
        print("Nessun dato da visualizzare.")
        return

    plt.figure(figsize=(8, 7))
    plt.imshow(dati, cmap=cmap, origin='lower')
    cbar = plt.colorbar(label='Intensità (ADU)', fraction=0.046, pad=0.04)
    plt.title(titolo, fontsize=14, fontweight='bold')
    plt.xlabel('Pixel X')
    plt.ylabel('Pixel Y')
    plt.tight_layout()
    plt.show()

visualizza_immagine(image_data)

## 5. Analisi statistica con istogramma

L'istogramma dei pixel mostra la distribuzione delle intensità nell'immagine. Questo ci permette di:
- Identificare il rumore di fondo
- Riconoscere sorgenti luminose
- Valutare la dinamica dell'immagine (range di valori)

In [ ]:
def analisi_istogramma(dati: np.ndarray, bins: int = 100):
    """Crea e visualizza un istogramma dei valori dei pixel.

    Args:
        dati: Array numpy dell'immagine
        bins: Numero di bins dell'istogramma
    """
    if dati is None:
        print("Nessun dato disponibile.")
        return

    pixel_flat = dati.flatten()
    print(f"Pixel totali: {len(pixel_flat)}")
    print(f"Forma appiattita: {pixel_flat.shape}")

    plt.figure(figsize=(10, 5))

    # Istogramma principale
    plt.subplot(1, 2, 1)
    counts, edges, _ = plt.hist(pixel_flat, bins=bins, color='steelblue',
                                 edgecolor='black', alpha=0.8)
    plt.xlabel('Intensità pixel')
    plt.ylabel('Frequenza')
    plt.title('Distribuzione delle intensità')
    plt.grid(True, alpha=0.3)

    # Istogramma zoomato (0-99° percentile)
    plt.subplot(1, 2, 2)
    limite_sup = np.percentile(pixel_flat, 99)
    plt.hist(pixel_flat, bins=bins, color='coral',
             edgecolor='black', alpha=0.8, range=(0, limite_sup))
    plt.xlabel('Intensità pixel (zoom 99° percentile)')
    plt.ylabel('Frequenza')
    plt.title('Dettaglio distribuzione (zoom)')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

analisi_istogramma(image_data)

## Riepilogo

In questo notebook abbiamo:
- Aperto e ispezionato un file FITS di M87
- Estratto e analizzato i dati dell'immagine
- Visualizzato l'immagine con falsi colori
- Analizzato la distribuzione statistica dei pixel tramite istogramma

M87 (Virgo A) è una galassia ellittica supergigante nella costellazione della Vergine, situata a circa 53 milioni di anni luce dalla Terra. Nel 2019 è stata la prima galassia ad essere fotografata direttamente nel suo buco nero supermassiccio centrale (M87*) dall'Event Horizon Telescope.